# GAEZ v5 - Suitability Class & Suitability Index

Reference: [GAEZ v5 Platform User Guide](https://github.com/un-fao/gaezv5/wiki/21.-GAEZ-v5-Platform-User-Guide) ("Download the data from Google Cloud Storage" section).

**Two variables**, both for all land (not just existing cropland):
- `RES05-SIX` = **Suitability Class** - categorical, 0-9 (S1-S4/marginal/not suitable/water/no data)
- `RES05-SXX` = **Suitability Index** - continuous score, 0-10000

Originally only pulled the Class, but it turns out the Class is too coarse to tell crops apart: 83% of cells have an *exact* tie in Class between the best and second-best crop (many cells have rice/maize/soybean literally all at the same class number). The continuous Index gives enough resolution to actually rank crops against each other per cell - see `05_eda.ipynb` section 11 for where this gets used.

**Config used** (kept the same across every crop/variable so results stay comparable):
- Period: `HP0120` (2001-2020, most recent historical window - not a future projection)
- Climate data source: `AGERA5`
- SSP: `HIST` (historical, not a climate-change scenario)
- Water & input level: `HRLM` (rain-fed, high input level of management)

Files are tiny either way (~0.4-0.5 MB for Class, ~2-3 MB for Index per crop) - nowhere near the multi-GB sizes originally expected. Note: despite the ~1km label shown in the FAO Hand-in-Hand web viewer, what's actually delivered through this bucket path is 5 arc-minute resolution (~9-10km at the equator) based on the raster's own metadata (2160x4320 global) - budget for that when aggregating onto a finer analysis grid.

In [1]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\download


In [2]:
from common import RAW_DIR, download_file, ensure_dir, get_logger

log = get_logger("gaez")

BASE_URL_TMPL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/{variable}/"
OUT_DIR = RAW_DIR / "gaez_v5"

# variable code -> (label used in the output filename, whether it's the categorical Class or continuous Index)
VARIABLES = {
    "RES05-SIX": "suitability_class",  # categorical, 0-9
    "RES05-SXX": "suitability_index",  # continuous, 0-10000
}

PERIOD = "HP0120"  # 2001-2020
CLIMATE_SOURCE = "AGERA5"
SSP = "HIST"  # historical, not a climate-change scenario
WATER_INPUT = "HRLM"  # rain-fed, high input level of management

# GAEZ v5 crop code -> local label (see CODES_CROP in
# GAEZv5_Themes_Sub-themes_Variables.xlsx for more crop codes, e.g. WPO=potato, SUC=sugar cane)
CROPS = {
    "RCW": "rice",     # Wetland rice
    "MZE": "maize",
    "CSV": "cassava",
    "SOY": "soybean",
}


def build_filename(variable: str, crop_code: str) -> str:
    return f"GAEZ-V5.{variable}.{PERIOD}.{CLIMATE_SOURCE}.{SSP}.{crop_code}.{WATER_INPUT}.tif"


def main() -> None:
    ensure_dir(OUT_DIR)
    for variable, out_label in VARIABLES.items():
        base_url = BASE_URL_TMPL.format(variable=variable)
        for crop_code, crop_label in CROPS.items():
            fname = build_filename(variable, crop_code)
            url = base_url + fname
            dest = OUT_DIR / f"{out_label}_{crop_label}_{crop_code}.tif"
            try:
                download_file(url, dest)
            except Exception as e:  # noqa: BLE001
                log.warning("Failed to download %s / %s (%s): %s", variable, crop_label, crop_code, e)

In [3]:
main()

suitability_class_rice_RCW.tif: 100%|██████████| 428k/428k [00:01<00:00, 228kB/s]  
[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_rice_RCW.tif
suitability_class_maize_MZE.tif: 100%|██████████| 551k/551k [00:02<00:00, 268kB/s]  
[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_maize_MZE.tif
suitability_class_cassava_CSV.tif: 100%|██████████| 452k/452k [00:00<00:00, 6.43MB/s]
[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_cassava_CSV.tif
suitability_class_soybean_SOY.tif: 100%|██████████| 544k/544k [00:02<00:00, 266kB/s]  
[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_soybean_SOY.tif
suitability_index_rice_RCW.tif: 100%|██████████| 1.90M/1.90M [00:02<00:00, 713kB/s] 
[download] Done: D:\Softwares\AnacondaProjects\GeoSuitabil

## Adding more crops

Look up the crop code in [GAEZv5_Themes_Sub-themes_Variables.xlsx](https://github.com/user-attachments/files/20961388/GAEZv5_Themes_Sub-themes_Variables.xlsx) (the `CODES_CROP` sheet), then add it to the `CROPS` dict above. E.g. `"WPO": "potato"` (white potato), `"SUC": "sugarcane"`.